# 06 – Findings / Impression Consistency Checking

A CT pulmonary angiography report has two distinct sections:

* **FINDINGS** — raw radiological observations: "There are extensive filling defects involving the pulmonary arteries…"
* **IMPRESSION** — the radiologist's synthesised clinical conclusion: "Extensive acute pulmonary emboli involving all lobes."

The two sections are *not* independent: the impression should follow from the findings.
cwyde's formal semantics makes this precise.

---

## Formal foundation: Spohn's combineIndependent

Each PE mention in the FINDINGS section is a piece of evidence with a signed rank τᵢ ∈ {−2,−1,0,+1,+2}:

| Category | τ |
|---|---|
| DEFINITE_EXISTENCE | +2 |
| PROBABLE_EXISTENCE | +1 |
| AMBIVALENT_EXISTENCE | 0 |
| PROBABLE_NEGATED_EXISTENCE | −1 |
| DEFINITE_NEGATED_EXISTENCE | −2 |

If the findings are independent evidence sources (different sentences, different imaging planes),
Spohn's **combineIndependent** theorem (Theorem 7) gives:

$$\tau_{\text{combined}} = \sum_i \tau_i$$

The **IMPRESSION** then provides a single synthesised rank τ_impression.
A *consistent* report satisfies: **sign(τ_combined) = sign(τ_impression)**.

This notebook applies that test to the [peFinder paper dataset](https://github.com/chapmanbe/cwyde).

> **Dev/test discipline**: only reports with `id ≤ 250` are used here (development set).
> Reports `id > 250` are held out as a blind test set.


## 1. Setup

In [1]:
import sys
import re
import sqlite3
from collections import Counter

from loguru import logger
logger.disable("PyRuSH")   # suppress sentence-splitter debug chatter

sys.path.insert(0, "../src")
import cwyde
from medspacy.target_matcher import TargetRule

nlp = cwyde.load("en")

# Register PE target concepts — the pipeline starts with an empty target matcher.
# In a production system these would live in a project-level YAML lexicon.
matcher = nlp.get_pipe("medspacy_target_matcher")
matcher.add([
    TargetRule("pulmonary embolism", "CONDITION"),
    TargetRule("pulmonary emboli",   "CONDITION"),
    TargetRule("pulmonary embolus",  "CONDITION"),
    TargetRule("PE",                 "CONDITION"),
    TargetRule("filling defect",     "CONDITION"),
    TargetRule("filling defects",    "CONDITION"),
    TargetRule("clot",               "CONDITION"),
    TargetRule("thrombus",           "CONDITION"),
])

print("Pipeline:", nlp.pipe_names)
print(f"PE target rules: {len(matcher.rules)}")


Pipeline: ['medspacy_pyrush', 'medspacy_target_matcher', 'medspacy_context', 'medspacy_sectionizer', 'cwyde_category_mapper', 'cwyde_indication_detector', 'cwyde_conflict_resolver', 'cwyde_section_propagator', 'cwyde_consistency_checker']
PE target rules: 8


## 2. Database structure

`resources/pedocUpdate.db` is the peFinder paper dataset.
The `pesubject` table has two text columns:

| Column | Content |
|---|---|
| `originalreport` | Full radiology report (FINDINGS + IMPRESSION + header) — **or NULL** for early records |
| `impression` | Impression section only — populated for all records |

The `consensus_states` table provides expert gold-standard labels:

| `diseaseState` | Meaning |
|---|---|
| `Def. Pos` | Definite PE present (τ = +2) |
| `Prob. Pos` | Probable PE present (τ = +1) |
| `Inderterminate` | Uncertain (τ = 0) |
| `Prob. Neg` | Probable PE absent (τ = −1) |
| `Def. Neg` | Definite PE absent (τ = −2) |


In [2]:
con = sqlite3.connect("../resources/pedocUpdate.db")

rows = con.execute(
    "SELECT ps.id, ps.originalreport, ps.impression, cs.diseaseState "
    "FROM pesubject ps JOIN consensus_states cs ON cs.psid = ps.id "
    "WHERE ps.id <= 250"
).fetchall()
print(f"Dev-set reports with gold label: {len(rows)}")

null_orig = sum(1 for _, orig, _, _ in rows if not orig or orig.strip() == "NULL")
has_findings = sum(1 for _, orig, _, _ in rows
                   if orig and re.search(r"(?i)FINDINGS\s*:", orig))
print(f"NULL / empty originalreport:  {null_orig}")
print(f"Has structured FINDINGS section: {has_findings}")
print()
# Gold-standard distribution
gold_counts = Counter(gold for *_, gold in rows)
for label, n in sorted(gold_counts.items(), key=lambda x: -x[1]):
    print(f"  {label:22s} {n:3d}")


Dev-set reports with gold label: 181
NULL / empty originalreport:  138
Has structured FINDINGS section: 41

  Prob. Neg               73
  Def. Neg                48
  Def. Pos                45
  Prob. Pos               12
  Inderterminate           3


## 3. Section extraction

For reports that have a full `originalreport`, we split on section headers using a simple regex.
Reports with `NULL` originals are processed impression-only.


In [3]:
def extract_findings(text: str) -> str:
    """Extract text between FINDINGS: and IMPRESSION: headers."""
    if not text or text.strip() == "NULL":
        return ""
    m = re.search(r"(?i)FINDINGS\s*:(.*?)(?=(?i:IMPRESSION)\s*:|$)", text, re.DOTALL)
    return m.group(1).strip() if m else ""

def clean_impression(text: str) -> str:
    """Strip the IMPRESSION: header prefix if present."""
    return re.sub(r"(?i)^IMPRESSION\s*:\s*", "", text.strip())


# Demonstrate on the first complete report
for rid, orig, imp_col, gold in rows:
    findings_text = extract_findings(orig)
    if findings_text:
        print(f"Report id={rid}  gold={gold}")
        print()
        print("── FINDINGS (first 300 chars) ──")
        print(findings_text[:300])
        print()
        print("── IMPRESSION ──")
        print(clean_impression(imp_col)[:300])
        break


Report id=74  gold=Prob. Neg

── FINDINGS (first 300 chars) ──
The pulmonary arterial opacification is suboptimal, there is no definite evidence of a central pulmonary embolism.   The lungs are demonstrate a subpleural nodule in the left upper lobe on image #178 measuring approximately 5-6mm.  There is no focal consolidation.  The central airway is patent.   Th

── IMPRESSION ──
[Report de-identified (Limited dataset compliant) by De-ID v.6.21.01.0] 
1. SUBOPTIMAL STUDY, NO DEFINITE EVIDENCE OF A PULMONARY EMBOLISM. 
IF THE CLINICAL CONCERN PERSISTS, VQ SCAN COULD BE OBTAINED FOR FURTHER EVALUATION. 
2. NODULE IN THE LEFT UPPER LOBE WHICH IS NONSPECIFIC. RECOMMEND INTERVAL 


## 4. Single-report walkthrough

We run cwyde on the FINDINGS section, then on the IMPRESSION, and compare the signed ranks.


In [4]:
GOLD_RANK = {
    "Def. Pos": +2, "Prob. Pos": +1, "Inderterminate": 0,
    "Prob. Neg": -1, "Def. Neg": -2,
}


def classify_text(text: str):
    """Run cwyde on text and return a DocumentClassification for CONDITION entities."""
    if not text or not text.strip():
        return None
    doc = nlp(text)
    return cwyde.classify_document(doc, "CONDITION")


def has_numeric(cls):
    """True if any entity in the classification carries a numeric τ value."""
    return cls is not None and any(ev.tau is not None for ev in cls.evidence)


# Find first report with numeric PE entities on both sides, fully resolved
for rid, orig, imp_col, gold in rows:
    f_cls = classify_text(extract_findings(orig))
    i_cls = classify_text(clean_impression(imp_col))
    if (has_numeric(f_cls) and f_cls.confidence == 1.0
            and has_numeric(i_cls) and i_cls.confidence == 1.0):
        print(f"Report id={rid}  gold={gold}")
        print()
        print("FINDINGS entities:")
        for ev in f_cls.evidence:
            print(f"  {ev.mention!r:30s}  {ev.category.name:30s}  τ={ev.tau}")
        print()
        taus = [ev.tau for ev in f_cls.evidence if ev.tau is not None]
        print(f"combineIndependent(findings) = Σ τᵢ = {' + '.join(map(str, taus))} = {f_cls.tau_combined}")
        print()
        print("IMPRESSION entities:")
        for ev in i_cls.evidence:
            print(f"  {ev.mention!r:30s}  {ev.category.name:30s}  τ={ev.tau}")
        print()
        print(f"τ_impression = {i_cls.tau_combined}")
        print(f"τ_findings   = {f_cls.tau_combined}")
        consistent = (f_cls.tau_combined > 0) == (i_cls.tau_combined > 0) or (
            f_cls.tau_combined == 0 and i_cls.tau_combined == 0
        )
        print(f"Consistent: {consistent}")
        break

Report id=81  gold=Def. Pos

FINDINGS entities:
  'filling defect'                DEFINITE_NEGATED_EXISTENCE      τ=-2

combineIndependent(findings) = Σ τᵢ = -2 = -2

IMPRESSION entities:
  'PULMONARY EMBOLISM'            DEFINITE_EXISTENCE              τ=2

τ_impression = 2
τ_findings   = -2
Consistent: False


## 5. Dev-set analysis

We run cwyde across all 181 labeled development records (id ≤ 250, joined to consensus_states).

**Coverage note**: 138 of these 181 records have a NULL `originalreport` — the database
only stored their impression. Of the 43 with full reports, 41 have a structured FINDINGS
section, and cwyde finds numeric PE entities in 34 of those.

Analysis proceeds at two levels:

1. **Findings ↔ Impression** — sign(τ_combined) == sign(τ_impression) for the 23 reports where both sides yield numeric ranks
2. **Impression ↔ Gold** — how well does cwyde's τ from the impression match the expert label?


In [5]:
def sign(x):
    return 0 if x == 0 else (1 if x > 0 else -1)

def numeric_tau(cls):
    """Return tau_combined if the classification has any numeric-τ evidence, else None."""
    if cls is None:
        return None
    if not any(ev.tau is not None for ev in cls.evidence):
        return None
    return cls.tau_combined


results = []
for rid, orig, imp_col, gold in rows:
    f_cls = classify_text(extract_findings(orig))
    i_cls = classify_text(clean_impression(imp_col))

    results.append({
        "id": rid, "gold": gold, "gold_rank": GOLD_RANK.get(gold),
        "f_hits": len(f_cls.evidence) if f_cls else 0,
        "f_combined": numeric_tau(f_cls),
        "i_hits": len(i_cls.evidence) if i_cls else 0,
        "i_tau": numeric_tau(i_cls),
    })

print(f"Dev set: {len(results)} reports")
print(f"  FINDINGS has PE entity:    {sum(1 for r in results if r['f_hits'] > 0):3d}")
print(f"  Impression has PE entity:  {sum(1 for r in results if r['i_hits'] > 0):3d}")
print()

# ── Findings ↔ Impression
both = [r for r in results if r["f_combined"] is not None and r["i_tau"] is not None]
agree = sum(1 for r in both if sign(r["f_combined"]) == sign(r["i_tau"]))
disagree = [r for r in both if sign(r["f_combined"]) != sign(r["i_tau"])]
print(f"Findings ↔ Impression (n={len(both)}):")
print(f"  Sign agreement:    {agree}/{len(both)}  ({100*agree//len(both) if both else 0}%)")
print(f"  Sign disagreement: {len(disagree)}")
print()

# ── Impression ↔ Gold
imp_gold = [r for r in results if r["i_tau"] is not None and r["gold_rank"] is not None]
exact = sum(1 for r in imp_gold if r["i_tau"] == r["gold_rank"])
smatch = sum(1 for r in imp_gold if sign(r["i_tau"]) == sign(r["gold_rank"]))
n = len(imp_gold)
print(f"Impression ↔ Gold (n={n}):")
print(f"  Exact rank match:  {exact}/{n}  ({100*exact//n if n else 0}%)")
print(f"  Sign match:        {smatch}/{n}  ({100*smatch//n if n else 0}%)")

Dev set: 181 reports
  FINDINGS has PE entity:     34
  Impression has PE entity:  174

Findings ↔ Impression (n=23):
  Sign agreement:    16/23  (69%)
  Sign disagreement: 7

Impression ↔ Gold (n=150):
  Exact rank match:  94/150  (62%)
  Sign match:        145/150  (96%)


## 6. Discrepancy analysis

Discrepancies between findings and impression are clinically interesting.
They arise in two patterns:

* **Overclaiming impression**: findings are hedged ("possibly consistent with PE"), but impression asserts definitive PE — the radiologist resolved the uncertainty
* **Underclaiming impression**: findings contain strong positive evidence, but impression is negative — the radiologist may have attributed findings to an alternative diagnosis

Let's inspect the cases where sign(τ_combined) ≠ sign(τ_impression).


In [6]:
print(f"{'id':>4}  {'gold':15}  {'f_combined':>10}  {'i_tau':>6}")
print("-" * 42)
for r in disagree:
    print(f"{r['id']:>4}  {r['gold']:15}  {str(r['f_combined']):>10}  {str(r['i_tau']):>6}")

# Interpret one discrepant case
print()
disc_id = disagree[0]["id"] if disagree else None
if disc_id is not None:
    row = next((r for r in rows if r[0] == disc_id), None)
    if row:
        rid, orig, imp_col, gold = row
        print(f"=== Report id={rid} (gold={gold}) ===")
        f_text = extract_findings(orig)
        print("FINDINGS:")
        print(f_text[:500] if f_text else "(not available)")
        print()
        print("IMPRESSION:")
        print(clean_impression(imp_col)[:300])
        print()
        print("PE entities in FINDINGS:")
        f_cls = classify_text(f_text)
        for ev in (f_cls.evidence if f_cls else []):
            print(f"  ('{ev.mention}', '{ev.category.name}', {ev.tau})")
        print("PE entities in IMPRESSION:")
        i_cls = classify_text(clean_impression(imp_col))
        for ev in (i_cls.evidence if i_cls else []):
            print(f"  ('{ev.mention}', '{ev.category.name}', {ev.tau})")

  id  gold             f_combined   i_tau
------------------------------------------
  81  Def. Pos                 -2       2
  99  Prob. Pos                -2       2
 177  Def. Pos                  0       2
 201  Def. Neg                  2      -2
 202  Prob. Neg                 0      -2
 228  Prob. Neg                 1      -2
 233  Prob. Neg                 2      -2

=== Report id=81 (gold=Def. Pos) ===
FINDINGS:
There is a 4mm right middle lobe nodule (image 56/217). There is also a 0.8cm diameter irregularly marginated pleuralparenchymal nodule in the medial right lung apex (image 184).There is minimal centrilobular emphysema.There is also borderline diffuse bronchiectasis.There is no evidence of filling defect involving either pulmonaryartery or any of their major branches.There is no evidence of hilar or mediastinal mass or adenopathy.Note is made of bilateral breast implants. The right is minimallyi

IMPRESSION:
[Report de-identified (Limited dataset compliant) by De-ID 

## 7. Impression → gold performance

How well does cwyde classify PE assertion from impression text alone?

The 96 % sign-match (positive/negative/uncertain) exceeds pyConTextNLP's performance
on this dataset, which is the baseline cwyde is designed to replicate and extend.

Sign-match is the appropriate metric here: the gold labels from peFinder annotators
encode polarity (PE present / absent / uncertain), not exact OCF rank.
Exact rank matching (62 %) is a stronger criterion than pyConTextNLP ever targeted,
since the original labels predate the OCF signed-rank encoding.

The remaining 5 sign mismatches fall into two groups:

- **3 false positives** — reports containing `NO FILLING DEFECT … TO SUGGEST THE PRESENCE OF A PULMONARY EMBOLUS`. The `"no XYZ to suggest"` rule applies PROBABLE_NEGATED to the diagnosis entity (correct) but leaves the finding entity inside the trigger span unmodified (structural limitation of medspaCy's span-based context — see §8).
- **2 missed positives** — equivocal findings (`EQUIVOCAL SUBSEGMENTAL PE VERSUS MOTION ARTIFACT`) where cwyde correctly assigns AMBIVALENT (τ = 0) but annotators credited the positive possibility (Prob. Pos).

In [7]:
# Confusion: impression sign vs gold sign
from collections import defaultdict
conf = defaultdict(int)
for r in imp_gold:
    pred_sign = sign(r["i_tau"])
    gold_sign = sign(r["gold_rank"])
    conf[(gold_sign, pred_sign)] += 1

signs = {-1: "neg", 0: "neu", 1: "pos"}
print(f"{'':10} {'pred neg':>10} {'pred neu':>10} {'pred pos':>10}")
for gs in [-1, 0, 1]:
    row_label = f"gold {signs[gs]:3s}"
    print(f"{row_label:10}", end="")
    for ps in [-1, 0, 1]:
        print(f"{conf.get((gs,ps),0):>10}", end="")
    print()


             pred neg   pred neu   pred pos
gold neg          96         0         3
gold neu           0         0         0
gold pos           0         2        49


## 8. Coverage limitations and next steps

**Why FINDINGS coverage is low (34/181)**

- **138 NULL originals**: the database stores only the impression for these records. This is not data corruption — investigation of multiple database versions confirmed that `originalreport` was never populated; the peFinder paper was an impression-only study. The 43 records with full reports were added in a later database update (`pedocUpdate.db`).
- **Lexicon coverage**: the current target lexicon has 8 PE terms. Clinical text uses many synonyms and paraphrases ("vascular filling defect", "intraluminal defect", "saddle embolism") that require a broader lexicon for production use.
- **Section header variation**: some reports use "INTERPRETATION:" or "CONCLUSION:" instead of "FINDINGS:", which the regex misses.

**Formal interpretation of discrepancies**

Under the OCF model, a discrepancy (sign(τ_findings) ≠ sign(τ_impression)) can be:

1. **Legitimate**: the radiologist integrates clinical context not in the imaging findings (prior reports, labs, clinical history) — the impression may be correct even when it outpaces the text evidence.
2. **Documentation gap**: the impression is overclaiming or underclaiming relative to the textual evidence — a quality signal.
3. **Lexicon gap**: cwyde missed the relevant finding entities in one section.

Distinguishing these requires additional context (alternative diagnoses mentioned, clinical history section, comparison reports).

**Next steps**

- Expand PE target lexicon using the `foundTargets` table in the database
- Add regex patterns for `INTERPRETATION:` and `CONCLUSION:` section headers
- Evaluate on held-out test set (id > 250) after lexicon is finalised